In [1]:
import matplotlib.pyplot as plt

def plot_loss(loss_history):
    plt.figure(figsize=(6,4))
    plt.plot(loss_history, marker='o', linewidth=2)
    plt.xlabel("Epoch", fontsize=12)
    plt.ylabel("BPR Loss", fontsize=12)
    plt.title("Training Loss per Epoch", fontsize=14)
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.show()

# 0. Dữ liệu đầu vào

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import LabelEncoder
from scipy.sparse import coo_matrix, diags
import math

def read_parquet(path: str | Path, **kw) -> pd.DataFrame:
    return pd.read_parquet(path, **kw)
metadata_train_path = "/kaggle/input/program-recomender/metadata_train.parquet"
df_metadata_train = read_parquet(metadata_train_path)
display(df_metadata_train.head(5))
display(df_metadata_train.shape)
metadata_val_path = "/kaggle/input/program-recomender/metadata_val.parquet"
df_metadata_val = read_parquet(metadata_val_path)
display(df_metadata_val.head(5))
display(df_metadata_val.shape)
metadata_test_path = "/kaggle/input/program-recomender/metadata_test.parquet"
df_metadata_test = read_parquet(metadata_test_path)
display(df_metadata_test.head(5))
display(df_metadata_test.shape)
logs_train_path = "/kaggle/input/program-recomender/logs_train.parquet"
df_logs_train = read_parquet(logs_train_path)
display(df_logs_train.head(5))
display(df_logs_train.shape)
logs_val_path = "/kaggle/input/program-recomender/logs_val.parquet"
df_logs_val = read_parquet(logs_val_path)
display(df_logs_val.head(5))
display(df_logs_val.shape)


,vsetv_id,tv_show_id,tv_show_category,director,genres_list,actors_list,start_time,prog_end,duration,year_of_production,iso_year,iso_week
0,3,0,Другое,None,"[None, None, None]",[],2020-03-09 06:00:00,2020-03-09 07:40:00,6000,None,2020,11
1,3,2400480,Инфо,None,"[общество и политика, None, None]",[],2020-03-09 07:40:00,2020-03-09 09:10:00,5400,1997,2020,11
2,3,700475,Развлечения,None,"[музыка, None, None]",[],2020-03-09 09:10:00,2020-03-09 11:30:00,8400,2010,2020,11
3,3,0,Сериалы,None,"[None, None, None]",[],2020-03-09 11:30:00,2020-03-09 19:30:00,28800,None,2020,11
4,3,2400480,Инфо,None,"[общество и политика, None, None]",[],2020-03-09 19:30:00,2020-03-09 20:15:00,2700,1997,2020,11


(1715975, 12)

,vsetv_id,tv_show_id,tv_show_category,director,genres_list,actors_list,start_time,prog_end,duration,year_of_production,iso_year,iso_week
0,3,90081012,Фильмы,Гильермо дель Торо,"[приключения, драма, фантазия]","[Салли Хокинс, Майкл Шеннон, Ричард Дженкинс]",2020-06-29 00:35:00,2020-06-29 02:50:00,8100,2017,2020,27
1,3,6500479,Познавательное,None,"[путешествия, None, None]",[],2020-06-29 02:50:00,2020-06-29 05:00:00,7800,2010,2020,27
2,3,2400480,Инфо,None,"[общество и политика, None, None]",[],2020-06-29 05:00:00,2020-06-29 06:30:00,5400,1997,2020,27
3,3,20088,Развлечения,None,"[информация (комплексная), None, None]",[],2020-06-29 06:30:00,2020-06-29 07:00:00,1800,1996-,2020,27
4,3,2400480,Инфо,None,"[общество и политика, None, None]",[],2020-06-29 07:00:00,2020-06-29 07:10:00,600,1997,2020,27


(430139, 12)

,vsetv_id,tv_show_id,tv_show_category,director,genres_list,actors_list,start_time,prog_end,duration
0,3,20088,Развлечения,None,"[информация (комплексная), None, None]",[],2020-07-27 06:30:00,2020-07-27 07:00:00,1800
1,3,2400480,Инфо,None,"[общество и политика, None, None]",[],2020-07-27 07:00:00,2020-07-27 07:10:00,600
2,3,20088,Развлечения,None,"[информация (комплексная), None, None]",[],2020-07-27 07:10:00,2020-07-27 08:00:00,3000
3,3,2400480,Инфо,None,"[общество и политика, None, None]",[],2020-07-27 08:00:00,2020-07-27 08:10:00,600
4,3,20088,Развлечения,None,"[информация (комплексная), None, None]",[],2020-07-27 08:10:00,2020-07-27 09:00:00,3000


(1295378, 9)

,user_id,tv_show_id,vsetv_id,start_time_view,end_time_view,duration_view,screen_time,session_id,iso_year,iso_week
0,2244466330591177,0,397,2020-03-09 08:45:54,2020-03-09 09:15:03,1749,0.501667,1,2020,11
1,2244466330591177,12002107,843,2020-03-09 09:15:43,2020-03-09 09:35:54,1211,0.403667,1,2020,11
2,2244466330591177,0,20,2020-03-09 09:36:00,2020-03-09 10:04:24,1704,0.258182,1,2020,11
3,2244466330591177,12002107,843,2020-03-12 08:42:22,2020-03-12 09:49:53,4051,1.000000,2,2020,11
4,2244466330591177,240081,5,2020-03-13 18:27:58,2020-03-13 19:35:11,4033,1.000000,3,2020,11


(1878969, 10)

,user_id,tv_show_id,vsetv_id,start_time_view,end_time_view,duration_view,screen_time,session_id,iso_year,iso_week
0,2244466330591177,90067940,805,2020-07-18 23:32:35,2020-07-19 02:33:11,10836,0.703788,24,2020,29
1,2244466330591177,10001261,805,2020-07-18 23:32:35,2020-07-19 02:33:11,10836,1.000000,24,2020,29
2,2244466330591177,10001261,805,2020-07-18 23:32:35,2020-07-19 02:33:11,10836,1.000000,24,2020,29
3,2244466330591177,12002806,597,2020-07-24 19:40:26,2020-07-24 20:01:39,1273,1.000000,25,2020,30
4,2244466330591177,10002647,805,2020-07-25 16:17:43,2020-07-25 16:51:50,2047,0.645667,26,2020,30


(525983, 10)

In [3]:
# ===== Remove invalid tv_show_id = 0 =====
def remove_invalid_tvshow(df: pd.DataFrame) -> pd.DataFrame:
    if "tv_show_id" in df.columns:
        return df[df["tv_show_id"] != 0].reset_index(drop=True)
    return df

# Metadata
df_metadata_train = remove_invalid_tvshow(df_metadata_train)
df_metadata_val   = remove_invalid_tvshow(df_metadata_val)
df_metadata_test  = remove_invalid_tvshow(df_metadata_test)

# Logs
df_logs_train = remove_invalid_tvshow(df_logs_train)
df_logs_val   = remove_invalid_tvshow(df_logs_val)
df_metadata_test = df_metadata_test[df_metadata_test["tv_show_id"] != 0].drop_duplicates("tv_show_id")

# 1. LightGCN (Collaborative Filtering Tower)  

## 1.1. Xây graph user–item cho LightGCN  
Map user_id → user_idx (0…n_users-1)  
Map tv_show_id → item_idx (0…n_items-1)  
weight (screen_time, overlap_s)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
print(device)

In [ ]:
def build_lightgcn_adj(num_users, num_items, logs):
    logs = logs.copy()
    # 1. Tính recency_days cho từng dòng log
    #    recency_days = số ngày từ lần xem đó đến lần xem mới nhất trong toàn bộ logs
    logs["recency_days"] = (
        logs["start_time_view"].max() - logs["start_time_view"]
    ).dt.total_seconds() / 86400.0
    # 2. Trọng số cho MỖI lần xem: screen_time * exp(-0.05 * recency_days)
    logs["edge_weight"] = (np.log1p(logs["screen_time"]) * np.exp(-0.05 * logs["recency_days"]))
    ui = (
        logs
        .groupby(["user_idx", "item_idx"], as_index=False)["edge_weight"]
        .sum()
        .rename(columns={"edge_weight": "w"})
    )
    users = ui["user_idx"].values
    items = ui["item_idx"].values + num_users     # shift item sang vùng node item
    weights = ui["w"].values.astype(np.float32)   # trọng số cạnh u-i
    rows = np.concatenate([users, items])
    cols = np.concatenate([items, users])
    data = np.concatenate([weights, weights])
    N = num_users + num_items
    adj = coo_matrix((data, (rows, cols)), shape=(N, N))
    deg = np.array(adj.sum(1)).flatten()
    deg[deg == 0] = 1
    deg_inv_sqrt = 1.0 / np.sqrt(deg)
    D = diags(deg_inv_sqrt)
    adj_norm = D @ adj @ D
    adj_norm = adj_norm.tocoo()
    indices = torch.from_numpy(np.vstack([adj_norm.row, adj_norm.col])).long()
    values = torch.from_numpy(adj_norm.data).float()
    return torch.sparse_coo_tensor(indices, values, (N, N))


## 1.2. Train LightGCN → CF Embedding cho User & Item  


In [6]:
class LightGCN(nn.Module):
    def __init__(self, num_users, num_items, dim=64, n_layers=3):
        super().__init__()
        self.U = nn.Embedding(num_users, dim)
        self.I = nn.Embedding(num_items, dim)
        nn.init.xavier_uniform_(self.U.weight)
        nn.init.xavier_uniform_(self.I.weight)
        self.n_layers = n_layers

    def propagate(self, adj):
        x = torch.cat([self.U.weight, self.I.weight], dim=0)
        embs = [x]
        for _ in range(self.n_layers):
            x = torch.sparse.mm(adj, x)
            embs.append(x)

        embs = torch.stack(embs, dim=1)
        final = embs.mean(dim=1)
        return final[:self.U.num_embeddings], final[self.U.num_embeddings:]


In [7]:
def bpr_loss(user_e, pos_e, neg_e):
    """
    Nếu pos > neg → sigmoid gần 1 → loss nhỏ
    Nếu pos < neg → sigmoid nhỏ → loss lớn
    """
    pos = (user_e * pos_e).sum(dim=-1)
    neg = (user_e * neg_e).sum(dim=-1)
    return -torch.log(torch.sigmoid(pos - neg) + 1e-10).mean()


In [8]:
class BPRSampler:
    def __init__(self, logs, num_items):
        self.pos = logs.groupby("user_idx")["item_idx"].apply(list).to_dict()
        self.num_items = num_items

    def sample(self, batch_size):
        """
        1. Chọn ngẫu nhiên 1 user
        2. Chọn ngẫu nhiên 1 positive item của user
        3. Random item cho tới khi chọn được item mà user chưa từng tương tác.
        """
        us, ps, ns = [], [], []
        for _ in range(batch_size):
            u = np.random.choice(list(self.pos.keys()))
            pos_list = self.pos[u]
            p = np.random.choice(pos_list)
            while True:
                n = np.random.randint(0, self.num_items)
                if n not in pos_list:
                    break
            us.append(u); ps.append(p); ns.append(n)
        return (
            torch.LongTensor(us),
            torch.LongTensor(ps),
            torch.LongTensor(ns),
        )


In [ ]:
def train_lightgcn(df_logs_train, adj, hparams, num_users, num_items):
    model = LightGCN(
        num_users=num_users,
        num_items=num_items,
        dim=hparams["emb_dim"],
        n_layers=hparams["n_layers"],
    ).to(device)
    sampler = BPRSampler(df_logs_train, num_items)
    optimizer = torch.optim.Adam(model.parameters(), lr=hparams["lr"])
    reg_lambda = hparams["reg_lambda"]
    epochs = 10
    batch_size = 4096
    loss_history = []
    for ep in range(epochs):
        model.train()
        losses = []
        for _ in range(200):
            u, p, n = sampler.sample(batch_size)
            u = u.to(device); p = p.to(device); n = n.to(device)

            U_emb, I_emb = model.propagate(adj)
            loss = bpr_loss(U_emb[u], I_emb[p], I_emb[n])

            # L2 regularization
            loss = loss + reg_lambda * (
                model.U.weight.norm(2).pow(2) + model.I.weight.norm(2).pow(2)
            )
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
            loss_history.append(loss.item())
        print(f"[LightGCN] epoch {ep+1}/{epochs}, loss={np.mean(losses):.4f}")

    return model

In [10]:
def get_cf_embedding(model, adj):
    with torch.no_grad():
        user_cf_emb, item_cf_emb = model.propagate(adj)
        user_cf_emb = F.normalize(user_cf_emb, p=2, dim=1)
        item_cf_emb = F.normalize(item_cf_emb, p=2, dim=1)
    return user_cf_emb, item_cf_emb


# 2. Content Encoder (MLP Tower)

## 2.1. Encode các thuộc tính

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import LabelEncoder
import hashlib
from collections import defaultdict

# 1) Stable hashing
def stable_hash_to_int(x: str, mod: int) -> int:
    # stable across runs/machines
    h = hashlib.md5(x.encode("utf-8")).hexdigest()
    return (int(h, 16) % (mod - 1)) + 1  # reserve 0 for padding

# 2) Build tensors + maps (NO training-time pandas)
def build_metadata_tensors(
    df_meta_all: pd.DataFrame,
    device: str,
    genre_dim: int = 128,
    actor_hash: int = 50000,
    max_actors: int = 5,
):
    df = df_meta_all.copy()
    df["director"] = df["director"].fillna("unknown").astype(str)
    df["tv_show_category"] = df["tv_show_category"].fillna("unknown").astype(str)
    df["vsetv_id"] = df["vsetv_id"].fillna("unknown").astype(str)
    df["genres_list"] = df["genres_list"].apply(
        lambda x: x if isinstance(x, list) and len(x) > 0 else ["unknown"]
    )
    df["actors_list"] = df["actors_list"].apply(
        lambda x: x if isinstance(x, list) else []
    )
    df["year_str"] = (
        df["year_of_production"]
        .astype(str)
        .str.extract(r"(\d{4})", expand=False)
        .fillna("unknown")
    )
    le = {}
    for col in ["director", "tv_show_category", "year_str", "vsetv_id"]:
        enc = LabelEncoder()
        df[col] = df[col].astype(str)
        df[col + "_id"] = enc.fit_transform(df[col])
        le[col] = enc

    # ---- duration normalize ----
    dur = df["duration"].astype(float)
    dur = dur.fillna(dur.median())
    dur_np = dur.to_numpy()
    dur_np = (dur_np - dur_np.mean()) / (dur_np.std() + 1e-6)

    # ---- genre multi-hot hashing into dense float32 [N, genre_dim] ----
    # (fast enough; no sklearn; no sparse->dense conversions)
    N = len(df)
    genre_feat = np.zeros((N, genre_dim), dtype=np.float32)
    for i, glist in enumerate(df["genres_list"].tolist()):
        if not glist:
            glist = ["unknown"]
        # simple multi-hot with normalization
        for g in glist:
            j = stable_hash_to_int(str(g), genre_dim) % genre_dim
            genre_feat[i, j] += 1.0
        s = genre_feat[i].sum()
        if s > 0:
            genre_feat[i] /= s

    # ---- actor ids [N, max_actors] (0 padding) ----
    actor_ids = np.zeros((N, max_actors), dtype=np.int64)
    for i, alist in enumerate(df["actors_list"].tolist()):
        alist = alist[:max_actors]
        for k, a in enumerate(alist):
            actor_ids[i, k] = stable_hash_to_int(str(a), actor_hash)

    # ---- to torch tensors (on device) ----
    t_dir  = torch.tensor(df["director_id"].to_numpy(), dtype=torch.long, device=device)
    t_cat  = torch.tensor(df["tv_show_category_id"].to_numpy(), dtype=torch.long, device=device)
    t_year = torch.tensor(df["year_str_id"].to_numpy(), dtype=torch.long, device=device)
    t_ch   = torch.tensor(df["vsetv_id_id"].to_numpy(), dtype=torch.long, device=device)

    t_genre = torch.tensor(genre_feat, dtype=torch.float32, device=device)
    t_actor = torch.tensor(actor_ids, dtype=torch.long, device=device)
    t_dur   = torch.tensor(dur_np, dtype=torch.float32, device=device)

    # ---- precompute maps for O(1) sampling ----
    director_to_items = defaultdict(list)
    genre_to_items = defaultdict(list)

    genres_lists = df["genres_list"].tolist()
    dir_ids = df["director_id"].to_numpy()

    for idx in range(N):
        director_to_items[int(dir_ids[idx])].append(idx)
        for g in genres_lists[idx]:
            genre_to_items[str(g)].append(idx)

    meta = {
        "df": df,  # keep for inference/export only, not used in training loop
        "genres_lists": genres_lists,
        "director_to_items": director_to_items,
        "genre_to_items": genre_to_items,
        "n_dir": int(df["director_id"].max()) + 1,
        "n_cat": int(df["tv_show_category_id"].max()) + 1,
        "n_year": int(df["year_str_id"].max()) + 1,
        "n_ch": int(df["vsetv_id_id"].max()) + 1,
        "le": le,
    }

    tensors = {
        "dir": t_dir,
        "cat": t_cat,
        "year": t_year,
        "ch": t_ch,
        "genre": t_genre,
        "actor": t_actor,
        "dur": t_dur,
    }
    return meta, tensors

## 2.2. Traning MLP

In [ ]:
class ContentEncoder(nn.Module):
    def __init__(
        self,
        n_dir, n_cat, n_year, n_ch,
        genre_dim=128,
        actor_hash=50000,
        max_actors=5,
        emb_dim=64,
    ):
        super().__init__()
        self.max_actors = max_actors
        self.dir_emb  = nn.Embedding(n_dir, 32)
        self.cat_emb  = nn.Embedding(n_cat, 16)
        self.year_emb = nn.Embedding(n_year, 8)
        self.ch_emb   = nn.Embedding(n_ch, 16)
        self.genre_fc = nn.Linear(genre_dim, 32)
        # padding_idx=0 => embedding(0) is always zero and not trained
        self.act_emb  = nn.Embedding(actor_hash, 32, padding_idx=0)
        in_dim = 32 + 16 + 8 + 16 + 32 + 32 + 1
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
            nn.Linear(256, emb_dim),
        )
    def forward(self, d, c, y, ch, g, a, dur):
        # a: [B, max_actors]
        a_emb = self.act_emb(a)         
        mask = (a != 0).unsqueeze(-1)   
        mask = mask.float()
        a_sum = (a_emb * mask).sum(dim=1)   
        cnt = mask.sum(dim=1).clamp(min=1.0) 
        a_mean = a_sum / cnt              
        g = self.genre_fc(g)  # [B, 32]
        dur = dur.unsqueeze(1)  # [B, 1]
        x = torch.cat([
            self.dir_emb(d),
            self.cat_emb(c),
            self.year_emb(y),
            self.ch_emb(ch),
            g,
            a_mean,
            dur
        ], dim=1)
        z = self.mlp(x)
        return z

In [13]:
# 4) Fast sampler for (anchor, positive, negatives)
#    Pos: same director OR share ≥1 genre
class PairSampler:
    def __init__(self, meta, seed=42):
        self.rng = np.random.default_rng(seed)
        self.N = len(meta["genres_lists"])
        self.genres_lists = meta["genres_lists"]
        self.director_to_items = meta["director_to_items"]
        self.genre_to_items = meta["genre_to_items"]

        # for faster neg sampling by director rejection
        self.dir_id = None  # optionally set externally for more constraints

    def sample_batch(self, batch_size: int):
        # returns anchor_idx, pos_idx, neg_idx: np arrays [B]
        a = self.rng.integers(0, self.N, size=batch_size, endpoint=False)

        p = np.empty(batch_size, dtype=np.int64)
        n = np.empty(batch_size, dtype=np.int64)

        for t, ai in enumerate(a):
            # ---- positive pool ----
            pos_pool = set()

            # same director
            # (director ids are already precomputed in director_to_items,
            #  but we need current director id -> we can reconstruct via membership:
            #  easiest: take director pools by scanning? no. We'll build a small helper:
            #  For production: store director_id array in sampler if you want extra speed.
            #  Here: approximate by using genre pools + random director pool choice:
            #  We'll do: pick positive from either director pool via cached lookup from df later.
            #  But to keep sampler pure, we will prefer genre-based positives (fast + good).
            for g in self.genres_lists[ai]:
                for idx in self.genre_to_items.get(str(g), []):
                    pos_pool.add(idx)

            if len(pos_pool) == 0:
                # fallback: random other item
                pi = int(self.rng.integers(0, self.N))
            else:
                pi = int(self.rng.choice(list(pos_pool)))

            # ---- negative: try avoid shared genres with anchor ----
            anchor_genres = set(map(str, self.genres_lists[ai]))
            for _ in range(20):
                ni = int(self.rng.integers(0, self.N))
                if ni == ai:
                    continue
                if anchor_genres.isdisjoint(map(str, self.genres_lists[ni])):
                    break
            n[t] = ni
            p[t] = pi

        return a, p, n

In [14]:
# 5) Losses: InfoNCE + CF distillation
def info_nce_loss(z_a, z_p, z_n, temperature=0.07):
    # z_*: [B, D], assumed normalized
    # logits: [B, 1 + B] (pos vs all negs in batch)
    B = z_a.size(0)

    pos = (z_a * z_p).sum(dim=1, keepdim=True)  # [B, 1]
    # use in-batch negatives: anchors vs neg embeddings
    # (z_n are "negative samples"; also act as a pool for all)
    neg = z_a @ z_n.t()  # [B, B]

    logits = torch.cat([pos, neg], dim=1) / temperature
    labels = torch.zeros(B, dtype=torch.long, device=z_a.device)  # class 0 is positive
    return F.cross_entropy(logits, labels)

def cf_distill_loss(z_content, e_cf, mode="mse"):
    # normalize outside if using cosine-style
    if mode == "mse":
        return F.mse_loss(z_content, e_cf)
    elif mode == "cosine":
        return 1.0 - (z_content * e_cf).sum(dim=1).mean()
    else:
        raise ValueError("mode must be 'mse' or 'cosine'")


In [ ]:
# 6) Train loop (AMP optional)
def train_content(
   content_model,
    tensors,
    item_cf_emb,
    metaidx_to_itemidx,
    train_meta_idx,
    epochs=3,
    batch_size=1024,
    lr=1e-3,
    device="cuda",
):
    device = device if torch.cuda.is_available() else "cpu"
    content_model = content_model.to(device)
    item_cf_emb = item_cf_emb.to(device)

    opt = torch.optim.AdamW(content_model.parameters(), lr=lr)
    content_model.train()

    # ---- convert train_meta_idx to torch on GPU (FIX) ----
    train_meta_idx = torch.tensor(train_meta_idx, device=device, dtype=torch.long)

    # meta_idx -> item_idx tensor aligned with train_meta_idx order
    meta_to_item = torch.tensor(
        [metaidx_to_itemidx[int(i)] for i in train_meta_idx.cpu().tolist()],
        device=device,
        dtype=torch.long,
    )

    def forward_by_idx(idx):
        return content_model(
            tensors["dir"][idx],
            tensors["cat"][idx],
            tensors["year"][idx],
            tensors["ch"][idx],
            tensors["genre"][idx],
            tensors["actor"][idx],
            tensors["dur"][idx],
        )
    loss_history = []
    for ep in range(epochs):
        perm = torch.randperm(train_meta_idx.numel(), device=device)
        losses = []
        for i in range(0, train_meta_idx.numel(), batch_size):
            b = perm[i:i+batch_size]
            m = train_meta_idx[b]            # torch on GPU ✅
            item_idx = meta_to_item[b]       # torch on GPU ✅
            z = F.normalize(forward_by_idx(m), dim=1)
            e_cf = F.normalize(item_cf_emb[item_idx], dim=1)
            loss = F.mse_loss(z, e_cf)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            losses.append(loss.item())
            loss_history.append(loss.item())
        print(f"[MLP] ep {ep+1}/{epochs}, loss={sum(losses)/len(losses):.4f}")

    content_model.eval()
    return content_model


In [16]:
# 7) Build content embedding (full batch)
@torch.no_grad()
def build_content_embedding_production(content_model, tensors, device="cuda", batch_size=4096):
    content_model.eval()
    N = tensors["dir"].shape[0]
    outs = []
    for s in range(0, N, batch_size):
        e = min(N, s + batch_size)
        idx = torch.arange(s, e, device=device)
        z = content_model(
            tensors["dir"][idx],
            tensors["cat"][idx],
            tensors["year"][idx],
            tensors["ch"][idx],
            tensors["genre"][idx],
            tensors["actor"][idx],
            tensors["dur"][idx],
        )
        outs.append(F.normalize(z, dim=1).detach().cpu())
    return torch.cat(outs, dim=0)

# 3. Hybrid Merge => Item embedding từ content và CF

In [17]:
import torch
import torch.nn.functional as F

def build_hybrid_embedding(
    item_content_emb,        # [num_meta, D]
    item_cf_emb,             # [num_items, D]
    metaidx_to_itemidx,      # dict: meta_idx -> item_idx
    num_items,
    alpha,
):
    device = item_cf_emb.device
    D = item_cf_emb.size(1)

    # 1) Project content embedding to ITEM SPACE
    content_item_space = torch.zeros((num_items, D), device=device)

    for meta_idx, item_idx in metaidx_to_itemidx.items():
        content_item_space[item_idx] = item_content_emb[meta_idx]

    # 2) Hybrid
    hybrid_item_emb = alpha * item_cf_emb + (1 - alpha) * content_item_space
    hybrid_item_emb = F.normalize(hybrid_item_emb, dim=1)

    return hybrid_item_emb


# 4. User Embedding từ lịch sử xem  


In [18]:
import numpy as np

def build_user_emb(
    df_logs_train,
    hybrid_item_emb,   # [num_items, D]  (ITEM SPACE)
):
    """
    Return:
        user_emb: dict {user_idx: np.array(D)}
    """
    hyb_np = hybrid_item_emb.detach().cpu().numpy()
    user_emb = {}

    # tính recency 1 lần
    logs = df_logs_train.copy()
    logs["recency_days"] = (
        logs["start_time_view"].max()
        - logs["start_time_view"]
    ).dt.total_seconds() / 86400.0

    for u, df_u in logs.groupby("user_idx"):
        item_idx_list = df_u["item_idx"].values.astype(int)

        if len(item_idx_list) == 0:
            continue

        # ---- lấy item embedding theo ITEM IDX (ĐÚNG) ----
        item_vecs = hyb_np[item_idx_list]   # [n_items, D]

        # ---- trọng số ----
        w = (
            df_u["screen_time"].values
            * np.exp(-0.05 * df_u["recency_days"].values)
        )

        if w.sum() == 0:
            w = np.ones_like(w)

        u_vec = np.average(item_vecs, axis=0, weights=w)
        u_vec = u_vec / (np.linalg.norm(u_vec) + 1e-8)

        user_emb[u] = u_vec

    return user_emb


# 5. Hàm recommend và padding

In [ ]:
def build_global_top(df_logs, meta_df, top_n=5):
    candidate_ids = set(meta_df["tv_show_id"].values)

    df = df_logs[
        df_logs["tv_show_id"].isin(candidate_ids)
        & (df_logs["tv_show_id"] != 0)
    ]

    global_top = (
        df.groupby("tv_show_id")["screen_time"]
          .sum()
          .sort_values(ascending=False)
          .index
          .tolist()
    )
    return global_top[:top_n]


def pad_to_k(shows, global_top, k=5, allowed_ids=None):
    # bỏ duplicate, giữ thứ tự
    seen = set()
    shows = [s for s in shows if not (s in seen or seen.add(s))]

    if len(shows) >= k:
        return shows[:k]

    # padding từ global_top, tránh trùng, và nếu có allowed_ids thì phải nằm trong allowed_ids
    extra = []
    for s in global_top:
        if s in seen:
            continue
        if (allowed_ids is not None) and (s not in allowed_ids):
            continue
        extra.append(s)
        seen.add(s)
        if len(shows) + len(extra) >= k:
            break

    return shows + extra

import torch
import torch.nn.functional as F

import torch
import torch.nn.functional as F

def recommend(
    user_idx,
    user_emb,          # dict {user_idx: np.array(D)}
    item_emb,          # tensor [num_items_candidate, D]
    meta_df,
    global_top,
    topk=5,
    seen_item_ids=None,
):
    candidate_ids = set(meta_df["tv_show_id"].values)
    base_shows = []

    # ---- CHECK USER ----
    if (user_idx is not None) and (user_idx in user_emb):
        u_vec = torch.tensor(
            user_emb[user_idx],
            device=item_emb.device,
            dtype=torch.float32,
        )
        u_vec = F.normalize(u_vec, dim=0)

        scores = torch.matmul(item_emb, u_vec)  # [num_items]
        k = min(topk * 3, item_emb.size(0))
        topk_idx = torch.topk(scores, k=k).indices.cpu().numpy()

        for idx in topk_idx:
            show_id = meta_df.iloc[idx]["tv_show_id"]
            if (seen_item_ids is not None) and (show_id in seen_item_ids):
                continue
            base_shows.append(show_id)
            if len(base_shows) >= topk:
                break

    # ---- PAD ----
    shows = pad_to_k(
        base_shows,
        global_top,
        k=topk,
        allowed_ids=candidate_ids,
    )
    return shows

In [21]:
def user_to_idx(user_id, user_le):
    """
    Chuyển user_id gốc sang user_idx (LabelEncoder).
    """
    try:
        return user_le.transform([user_id])[0]
    except ValueError:
        return None

# 6. Huấn luyện

## 6.1. Tranning với từng bộ tham số

In [21]:
import numpy as np
import torch

def run_trial(
    hparams,
    df_logs_train,
    df_meta_all,
    meta_val,
    actual_dict_filtered,
    adj,
    itemidx_to_metaidx,
    metaidx_to_itemidx,
    user_le,
    meta,
    tensors,
    num_users,
    num_items,
    topk=5,
    device="cuda",
    # content training fixed (hoặc đưa vào hparams cũng được)
    content_epochs=3,
    content_steps_per_epoch=1000,
    content_batch_size=512,
    content_beta=0.2,
    content_lr=1e-3,
    content_temp=0.07,
):
    device = device if torch.cuda.is_available() else "cpu"

    # 1) TRAIN LIGHTGCN (CF)
    model_cf = train_lightgcn(
        df_logs_train=df_logs_train,
        adj=adj,
        hparams=hparams,          # dùng hparams["emb_dim"], ["n_layers"], ["lr"], ["reg_lambda"]
        num_users=num_users,
        num_items=num_items,
        device=device,
    )
    user_cf_emb, item_cf_emb = get_cf_embedding(model_cf, adj)   # shapes: [U,D], [I,D]
    item_cf_emb = item_cf_emb.to(device)

    # 2) TRAIN CONTENT MODEL (Semantic + CF-align)  (emb_dim phải theo trial)
    content_model = ContentEncoder(
        n_dir=meta["n_dir"],
        n_cat=meta["n_cat"],
        n_year=meta["n_year"],
        n_ch=meta["n_ch"],
        genre_dim=tensors["genre"].shape[1],
        actor_hash=50000,
        max_actors=tensors["actor"].shape[1],
        emb_dim=hparams["emb_dim"],      # <<< IMPORTANT
    ).to(device)

    content_model = train_content_model_production(
        content_model,
        meta=meta,
        tensors=tensors,
        item_cf_emb=item_cf_emb,                 # [num_items, emb_dim] ideally same dim
        metaidx_to_itemidx=metaidx_to_itemidx,
        train_meta_idx=hparams["train_meta_idx"],  # pass in via hparams for consistency
        epochs=content_epochs,
        steps_per_epoch=content_steps_per_epoch,
        batch_size=content_batch_size,
        beta=content_beta,
        lr=content_lr,
        temperature=content_temp,
        distill_mode="mse",
        device=device,
        use_amp=True,
    )

    item_content_emb = build_content_embedding_production(content_model, tensors, device=device)
    item_content_emb = item_content_emb.to(device)

    # 3) HYBRID ITEM EMBEDDING  (alpha theo trial)
    # Lưu ý: item_cf_emb index theo item_idx; item_content_emb index theo meta_idx
    # => cần map meta_idx -> item_idx để align
    # Cách đơn giản: tạo content_in_item_space [num_items, D]
    hybrid_emb_final = build_hybrid_embedding(
        item_content_emb=item_content_emb,
        item_cf_emb=item_cf_emb,
        metaidx_to_itemidx=metaidx_to_itemidx,
        num_items=num_items,
        alpha=best_hparams["alpha"],
    )

    # 4) USER EMBEDDING (từ logs_train + hybrid item emb)
    user_emb = build_user_emb(
        df_logs_train=df_logs_train,
        hybrid_emb=hybrid_item_emb_itemspace,
        itemidx_to_metaidx=itemidx_to_metaidx,   # nếu build_user_emb cần meta space thì bạn đổi lại hàm
    )

    # 5) ITEM EMBEDDING FOR VALIDATION SET
    meta_val = meta_val[meta_val["tv_show_id"] != 0].reset_index(drop=True)

    # map meta_val item_idx -> embedding trong itemspace
    val_item_idx = meta_val["item_idx"].values
    val_item_idx = [i for i in val_item_idx if i < num_items]
    item_val_emb = hybrid_item_emb_itemspace[val_item_idx]   # [V, D]

    # 6) GLOBAL TOP (POPULARITY)
    global_top_val = build_global_top(df_logs_train, meta_val)

    # 7) PREDICT
    pred_dict = {}
    for user_id in actual_dict_filtered:
        u_idx = user_to_idx(user_id, user_le)
        if u_idx is None:
            continue
        pred_dict[user_id] = recommend(
            user_idx=u_idx,
            user_emb=user_emb,
            item_emb=item_val_emb,
            meta_df=meta_val,
            global_top=global_top_val,
            topk=topk,
        )

    # 8) METRICS
    map5 = mapk(actual_dict_filtered, pred_dict, k=topk)
    recall5 = mean_recall_at_k(actual_dict_filtered, pred_dict, k=topk)
    ndcg5 = mean_ndcg_at_k(actual_dict_filtered, pred_dict, k=topk)

    return {
        "hparams": hparams,
        "map@5": float(map5),
        "recall@5": float(recall5),
        "ndcg@5": float(ndcg5),
        "hybrid_item_emb": hybrid_item_emb_itemspace.detach().cpu(),
        "user_cf_emb": user_cf_emb.detach().cpu() if torch.is_tensor(user_cf_emb) else user_cf_emb,
    }

## 6.2. Đánh giá trên tập val


In [23]:
def apk(actual, predicted, k=5):
    predicted = predicted[:k]
    score = 0.0
    num_hits = 0.0

    for i, p in enumerate(predicted):
        if p in actual:
            num_hits += 1.0
            score += num_hits / (i + 1.0)

    if not actual:
        return 0.0
    return score / min(len(actual), k)

def mapk(actual_dict, pred_dict, k=5):
    return np.mean([apk(actual_dict[u], pred_dict.get(u, []), k) for u in actual_dict])

def recall_at_k(actual, predicted, k=5):
    predicted = predicted[:k]
    return len(set(actual) & set(predicted)) / len(actual) if actual else 0

def mean_recall_at_k(actual_dict, pred_dict, k=5):
    return np.mean([recall_at_k(actual_dict[u], pred_dict[u], k) for u in actual_dict])

def ndcg_at_k(actual, predicted, k=5):
    predicted = predicted[:k]
    dcg = sum([1.0 / math.log2(i+2) for i,p in enumerate(predicted) if p in actual])
    idcg = sum([1.0 / math.log2(i+2) for i in range(min(len(actual), k))])
    return dcg / idcg if idcg > 0 else 0

def mean_ndcg_at_k(actual_dict, pred_dict, k=5):
    return np.mean([ndcg_at_k(actual_dict[u], pred_dict[u], k) for u in actual_dict])


In [34]:
# ===== 1) Filter logs_val theo metadata_val =====
val_show_ids = set(df_metadata_val["tv_show_id"].unique())

df_logs_val = df_logs_val[
    df_logs_val["tv_show_id"].isin(val_show_ids)
].copy()

# ===== 2) Compute recency =====
df_logs_val["recency_days"] = (
    df_logs_val["start_time_view"].max()
    - df_logs_val["start_time_view"]
).dt.total_seconds() / 86400.0

# ===== 3) Interaction score =====
df_logs_val["score"] = (
    df_logs_val["screen_time"]
    * np.exp(-0.05 * df_logs_val["recency_days"])
)

# ===== 4) Build ground truth =====
actual_dict = (
    df_logs_val
    .groupby(["user_id", "tv_show_id"])["score"]
    .sum()
    .reset_index()
    .sort_values(["user_id", "score"], ascending=[True, False])
    .groupby("user_id")
    .head(5)
    .groupby("user_id")["tv_show_id"]
    .apply(list)
    .to_dict()
)

print("Số user có ground truth:", len(actual_dict))

# ===== 5) Filter user: appeared in TRAIN only =====
train_users = set(df_logs_train["user_id"].unique())

MIN_ITEMS = 5
actual_dict_filtered = {
    u: items
    for u, items in actual_dict.items()
    if (u in train_users) and (len(items) >= MIN_ITEMS)
}

print("Số user sau khi filter:", len(actual_dict_filtered))

Số user có ground truth: 4720
Số user sau khi filter: 4373


## 6.3. Thực hiện training -> val

In [ ]:
# ===== Metadata =====
df_meta_all = pd.concat(
    [df_metadata_train, df_metadata_val],
    ignore_index=True
)

item_le = LabelEncoder()
df_meta_all["item_idx"] = item_le.fit_transform(df_meta_all["tv_show_id"])
df_logs_train["item_idx"] = item_le.transform(df_logs_train["tv_show_id"])

# meta_idx = row index of df_meta_all
metaidx_to_itemidx = dict(
    zip(df_meta_all.index.values, df_meta_all["item_idx"].values)
)

# ===== Logs =====
user_le = LabelEncoder()
df_logs_train["user_idx"] = user_le.fit_transform(df_logs_train["user_id"])

uid_map = {u: i for i, u in enumerate(user_le.classes_)}
df_logs_val["user_idx"] = df_logs_val["user_id"].map(uid_map)

num_items = df_meta_all["item_idx"].nunique()
num_users = df_logs_train["user_idx"].nunique()

# ===== Graph =====
adj = build_lightgcn_adj(num_users, num_items, df_logs_train)
adj = adj.to(device)

# ===== Content metadata tensors =====
meta, tensors = build_metadata_tensors(df_meta_all, device=device)

In [40]:
import random

search_space = {
    "emb_dim":    [64],
    "n_layers":   [2, 3],
    "lr":         [1e-3, 5e-3, 5e-4],
    "alpha":      [0.2, 0.4, 0.6],
    "reg_lambda": [1e-5, 1e-4, 1e-3],
}

def sample_hparams(space, train_meta_idx):
    hp = {
        "emb_dim": random.choice(space["emb_dim"]),
        "n_layers": random.choice(space["n_layers"]),
        "lr": random.choice(space["lr"]),
        "alpha": random.choice(space["alpha"]),
        "reg_lambda": random.choice(space["reg_lambda"]),
        "train_meta_idx": np.array(train_meta_idx, dtype=np.int64),
    }
    return hp

best = None
N_TRIALS = 10

for trial in range(N_TRIALS):
    hp = sample_hparams(search_space, train_meta_idx)

    print("\n===== Trial", trial+1, "/", N_TRIALS, "=====")
    print("Hyperparams:", {k:v for k,v in hp.items() if k != "train_meta_idx"})

    out = run_trial(
        hparams=hp,
        df_logs_train=df_logs_train,
        df_meta_all=df_meta_all,
        meta_val=df_metadata_val,
        actual_dict_filtered=actual_dict_filtered,
        adj=adj,
        itemidx_to_metaidx=itemidx_to_metaidx,
        metaidx_to_itemidx=metaidx_to_itemidx,
        user_le=user_le,
        meta=meta,
        tensors=tensors,
        num_users=num_users,
        num_items=num_items,
        topk=5,
        device=device,
    )

    print(f"MAP@5    = {out['map@5']:.4f}")
    print(f"Recall@5 = {out['recall@5']:.4f}")
    print(f"NDCG@5   = {out['ndcg@5']:.4f}")

    if (best is None) or (out["map@5"] > best["map@5"]):
        best = out

print("\n>>> BEST HP:", {k:v for k,v in best["hparams"].items() if k != "train_meta_idx"})
print(f">>> BEST MAP@5   : {best['map@5']:.4f}")
print(f">>> BEST Recall@5: {best['recall@5']:.4f}")
print(f">>> BEST NDCG@5  : {best['ndcg@5']:.4f}")

best_hparams = best["hparams"]
best_hybrid_emb = best["hybrid_item_emb"]
best_user_cf_emb = best["user_cf_emb"]


## 6.4. Training lại sau khi tìm được bộ tham số tốt nhất

In [25]:
# ===== Metadata =====
df_meta_all = pd.concat(
    [df_metadata_train, df_metadata_val, df_metadata_test],
    ignore_index=True
)
df_logs_all = pd.concat([df_logs_train, df_logs_val], ignore_index=True)

item_le = LabelEncoder()
df_meta_all["item_idx"] = item_le.fit_transform(df_meta_all["tv_show_id"])
df_logs_all["item_idx"] = item_le.transform(df_logs_all["tv_show_id"])

# meta_idx = row index of df_meta_all
metaidx_to_itemidx = dict(
    zip(df_meta_all.index.values, df_meta_all["item_idx"].values)
)
itemidx_to_metaidx = (
    df_meta_all.reset_index()
    .sort_values("index")                # giữ thứ tự gốc
    .drop_duplicates("item_idx", keep="first")
    .set_index("item_idx")["index"]
    .to_dict()
)

# ===== Logs (train + val) =====
user_le = LabelEncoder()
df_logs_all["user_idx"] = user_le.fit_transform(df_logs_all["user_id"])
df_logs_all["item_idx"] = item_le.transform(df_logs_all["tv_show_id"])

num_items = df_meta_all["item_idx"].nunique()
num_users = df_logs_all["user_idx"].nunique()

print("num_items =", num_items)
print("num_users =", num_users)

num_items = 9670
num_users = 4876


In [ ]:
best_hparams = {
    "emb_dim": 64,
    "n_layers": 3,
    "lr": 1e-3,
    "alpha": 0.4,
    "reg_lambda": 1e-5,
}


# 1) Light GCN
adj = build_lightgcn_adj(num_users, num_items, df_logs_all)
adj = adj.to(device)

model_cf = train_lightgcn(
    df_logs_all,
    adj,
    best_hparams,
    num_users,
    num_items,
)

user_cf_emb, item_cf_emb = get_cf_embedding(model_cf, adj)
item_cf_emb = item_cf_emb.to(device)




In [ ]:
# 2) MLP
meta, tensors = build_metadata_tensors(df_meta_all, device=device)
# ===== Content training config =====
content_epochs = 5
content_steps_per_epoch = 1500
content_batch_size = 512
content_beta = 0.2
content_lr = 1e-3
content_temp = 0.07

content_model = ContentEncoder(
    n_dir=meta["n_dir"],
    n_cat=meta["n_cat"],
    n_year=meta["n_year"],
    n_ch=meta["n_ch"],
    genre_dim=tensors["genre"].shape[1],
    actor_hash=50000,
    max_actors=tensors["actor"].shape[1],
    emb_dim=best_hparams["emb_dim"],
).to(device)

train_meta_idx = np.arange(len(df_meta_all))

content_model = train_content(
    content_model=content_model,
    tensors=tensors,
    item_cf_emb=item_cf_emb,
    metaidx_to_itemidx=metaidx_to_itemidx,
    train_meta_idx=train_meta_idx,   # numpy ok, hàm sẽ tự convert
    epochs=content_epochs,
    batch_size=content_batch_size,
    lr=content_lr,
    device=device,
)

item_content_emb = build_content_embedding_production(
    content_model, tensors, device=device
).to(device)


# 3) hybrid => item_embedding
hybrid_emb_final = build_hybrid_embedding(
    item_content_emb=item_content_emb,
    item_cf_emb=item_cf_emb,
    metaidx_to_itemidx=metaidx_to_itemidx,
    num_items=num_items,
    alpha=best_hparams["alpha"],
)

# 4) user_embedding
user_emb = build_user_emb(df_logs_all, hybrid_emb_final)

In [61]:
itemidx_to_metaidx = (
    df_meta_all.reset_index()
    .sort_values("index")                # giữ thứ tự gốc
    .drop_duplicates("item_idx", keep="first")
    .set_index("item_idx")["index"]
    .to_dict()
)

# 7. Recommemd trên tập test

In [66]:
# ===== Prepare test metadata & embeddings =====
meta_test = df_metadata_test.copy()
meta_test = meta_test[meta_test["tv_show_id"] != 0].reset_index(drop=True)
meta_test["item_idx"] = item_le.transform(meta_test["tv_show_id"])

# hybrid_emb_final: [num_items, dim]  (ITEM SPACE)
item_test_emb = hybrid_emb_final[
    meta_test["item_idx"].values
].to(device)

# ===== Users to predict =====
submission_input = pd.read_csv("/kaggle/input/program-recomender/submission (3).csv")
users_to_predict = submission_input["user_id"].tolist()

print("Số user cần predict:", len(users_to_predict))

# ===== Global popularity for TEST =====
global_top_test = build_global_top(df_logs_all, meta_test, top_n=50) 

# ===== Recommend =====
rows = []

for user_id in users_to_predict:
    u_idx = user_to_idx(user_id, user_le)

    # cold-start user
    if (u_idx is None) or (u_idx not in user_emb):
        shows = global_top_test[:5]
    else:
        shows = recommend(
            user_idx=u_idx,
            user_emb=user_emb,          # dict {user_idx: vector}
            item_emb=item_test_emb,     # tensor [num_test_items, dim]
            meta_df=meta_test,          # align với item_test_emb
            global_top=global_top_test,
            topk=5,
            seen_item_ids=None,         # hoặc set item user đã xem nếu muốn
        )

    rows.append([user_id, " ".join(map(str, shows))])

# ===== Save submission =====
submission = pd.DataFrame(rows, columns=["user_id", "tv_show_id"])
submission.to_csv("submission.csv", index=False)


Số user cần predict: 1260


In [47]:
pred_ids = set(" ".join(submission["tv_show_id"]).split())
meta_ids = set(df_metadata_test["tv_show_id"].astype(str))

print("ID không hợp lệ:", pred_ids - meta_ids)


ID không hợp lệ: set()


In [68]:
print(global_top_test)

[2400480, 240081, 12001682, 20088, 2400508, 2400467, 12002955, 12002856, 12001734, 6500479, 12002805, 6700482, 12002896, 1100411, 200344, 10002514, 200337, 500281, 500331, 700389, 500315, 12002201, 2500413, 12002378, 5200303, 200432, 200352, 12001732, 6200371, 1000376, 400361, 12002355, 5900256, 400335, 200469, 10001336, 10002451, 6000483, 10002893, 1000589, 12002806, 5800509, 10002515, 2500430, 10001261, 700369, 1000381, 10003159, 200506, 400426]


In [69]:
submission.head(10) 

,user_id,tv_show_id
0,8377619604347126107,240081 2400480 12001682 20088 2400508
1,8381667675275833309,2400480 240081 12001682 20088 2400508
2,8387147770138767246,12002856 2400480 240081 12001682 20088
3,8397181578236218580,90074315 90078673 2400480 240081 12001682
4,8404698046253197367,5200464 2400480 240081 12001682 20088
5,8431897575013155337,90080795 90080509 90033363 2400480 240081
6,8437234709656005918,12001682 2400480 240081 20088 2400508
7,8441113802219662639,90062637 2400480 240081 12001682 20088
8,8443187512168717371,240081 2400480 12001682 20088 2400508
9,8449544243572273063,1000376 2400480 240081 12001682 20088
